In [1]:
{
 "cells": [
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "# Steam Analytics Dashboard\n",
    "\n",
    "This notebook covers sentiment prediction using game metrics and provides similar game suggestions using hybrid filtering (genre, price, upvotes).\n"
   ]
  },
  {
   "cell_type": "code",
   "metadata": {},
   "outputs": [],
   "source": [
    "import ast\n",
    "import os\n",
    "from collections import Counter\n",
    "from pathlib import Path\n",
    "import joblib\n",
    "import matplotlib.pyplot as plt\n",
    "import numpy as np\n",
    "import pandas as pd\n",
    "import seaborn as sns\n",
    "from sklearn.ensemble import RandomForestClassifier\n",
    "from sklearn.linear_model import LogisticRegression\n",
    "from sklearn.metrics import accuracy_score, classification_report, confusion_matrix\n",
    "from sklearn.metrics.pairwise import cosine_similarity\n",
    "from sklearn.model_selection import GridSearchCV, train_test_split\n",
    "from sklearn.pipeline import Pipeline\n",
    "from sklearn.preprocessing import MultiLabelBinarizer, StandardScaler\n",
    "from sklearn.tree import DecisionTreeClassifier\n",
    "\n",
    "TARGET_CSV = \"games_march2025_cleaned.csv\"\n",
    "LOOKUP_DIRS = [Path.cwd(), Path(\"/content\")]\n",
    "CATEGORIES = [\"Negative\", \"Mixed\", \"Positive\"]\n",
    "\n",
    "def locate_file(fname):\n",
    "    for d in LOOKUP_DIRS:\n",
    "        if (d / fname).exists(): return d / fname\n",
    "    return LOOKUP_DIRS[0] / fname\n",
    "\n",
    "def map_sentiment(score):\n",
    "    if score >= 70: return \"Positive\"\n",
    "    if score >= 40: return \"Mixed\"\n",
    "    return \"Negative\"\n",
    "\n",
    "def display_metrics(y_real, y_pred, title=\"\"):\n",
    "    if title:\n",
    "        print(f\"\\n--- {title} ---\")\n",
    "    print(\"Accuracy:\", round(accuracy_score(y_real, y_pred), 4))\n",
    "    print(classification_report(y_real, y_pred, zero_division=0))\n"
   ]
  },
  {
   "cell_type": "code",
   "metadata": {},
   "outputs": [],
   "source": [
    "dataset_file = locate_file(TARGET_CSV)\n",
    "steam_data = pd.read_csv(dataset_file, engine=\"python\", on_bad_lines=\"skip\")\n",
    "print(\"Data Loaded! Shape:\", steam_data.shape)\n",
    "display(steam_data.head(3))\n"
   ]
  },
  {
   "cell_type": "code",
   "metadata": {},
   "outputs": [],
   "source": [
    "modeling_df = steam_data[steam_data[\"pct_pos_total\"].between(0, 100)].copy()\n",
    "modeling_df[\"target_label\"] = modeling_df[\"pct_pos_total\"].apply(map_sentiment)\n",
    "\n",
    "features = [\"price\", \"required_age\", \"windows\", \"mac\", \"linux\", \"average_playtime_forever\", \"discount\"]\n",
    "X_data = modeling_df[features].fillna(0).copy()\n",
    "y_data = modeling_df[\"target_label\"].copy()\n",
    "\n",
    "X_train, X_val, y_train, y_val = train_test_split(X_data, y_data, test_size=0.2, random_state=99, stratify=y_data)\n"
   ]
  },
  {
   "cell_type": "code",
   "metadata": {},
   "outputs": [],
   "source": [
    "rf_clf = RandomForestClassifier(n_estimators=150, max_depth=12, class_weight=\"balanced\", random_state=99)\n",
    "rf_clf.fit(X_train, y_train)\n",
    "preds = rf_clf.predict(X_val)\n",
    "display_metrics(y_val, preds, \"Random Forest Baseline\")\n"
   ]
  }
 ],
 "metadata": {
  "kernelspec": {
   "display_name": "Python 3",
   "language": "python",
   "name": "python3"
  },
  "language_info": {
   "name": "python",
   "pygments_lexer": "ipython3"
  }
 },
 "nbformat": 4,
 "nbformat_minor": 5
}

{'cells': [{'cell_type': 'markdown',
   'metadata': {},
   'source': ['# Steam Analytics Dashboard\n',
    '\n',
    'This notebook covers sentiment prediction using game metrics and provides similar game suggestions using hybrid filtering (genre, price, upvotes).\n']},
  {'cell_type': 'code',
   'metadata': {},
   'outputs': [],
   'source': ['import ast\n',
    'import os\n',
    'from collections import Counter\n',
    'from pathlib import Path\n',
    'import joblib\n',
    'import matplotlib.pyplot as plt\n',
    'import numpy as np\n',
    'import pandas as pd\n',
    'import seaborn as sns\n',
    'from sklearn.ensemble import RandomForestClassifier\n',
    'from sklearn.linear_model import LogisticRegression\n',
    'from sklearn.metrics import accuracy_score, classification_report, confusion_matrix\n',
    'from sklearn.metrics.pairwise import cosine_similarity\n',
    'from sklearn.model_selection import GridSearchCV, train_test_split\n',
    'from sklearn.pipeline import Pip